# បង្កើតសំឡេងខ្មែរផ្ទាល់ខ្លួន — Train your own Khmer voice (free Google Colab)

Fine-tune Meta's Khmer text-to-speech model so it speaks **in your voice**, then use it
in **Sonora Khmer Studio** — offline, no subscription.

**What you need**
| | |
|---|---|
| Google account | for Colab (free tier, T4 GPU) |
| Hugging Face account | free — you need a **write** token: https://huggingface.co/settings/tokens |
| Your recordings + Khmer text | see the next cell; or use the free OpenSLR Khmer set |

**How long:** the recipe can produce a usable voice from ~80–150 short clips in about
20 minutes of GPU time. 20–60 minutes of your voice gives a much steadier narrator.

**Be honest with yourself about the starting point:** the base MMS Khmer model is
*robotic but faithful* (it reads Khmer correctly). Fine-tuning on your own clean,
single-speaker recordings is what turns it into a natural, personal voice.


## 0 · Your recordings — what good training data looks like

* **one speaker only** (you). Mixing voices produces a smeared, blurry voice.
* **1–20 seconds per clip** (the recipe's window), 16 kHz or higher, mono.
* **clean**: no music, no room echo, no background talk; a quiet room + phone
  close to your mouth beats a fancy mic in a big room.
* **text**: Khmer script, matching the audio word for word.

No text yet? Two options:
1. Use the free **OpenSLR Khmer** set (downloaded in this notebook) — a general male
   Khmer voice, not yours.
2. Record 150–300 short clips (2–4 sentences each) and let a Khmer speaker or a
   Khmer ASR model draft the transcripts, then correct them.

Use the **prepare_khmer_dataset.py** tool that ships with Sonora Khmer Studio
(`khmer-colab/prepare_khmer_dataset.py`) to turn a folder of  clips + text into the exact
dataset shape this notebook expects — it checks duration, silence, Khmer script and
sample rate for you.


In [ ]:
# 1 · Is this a GPU runtime?  (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n!! No GPU. Menu: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU")

In [ ]:
# 2 · Install the training recipe (ylacombe/finetune-hf-vits — the official VITS/MMS one)
%cd /content
!rm -rf finetune-hf-vits
!git clone --depth 1 https://github.com/ylacombe/finetune-hf-vits.git
%cd finetune-hf-vits
!pip install -q -r requirements.txt
!pip install -q datasets soundfile librosa

# Cython monotonic alignment search — REQUIRED, the pure-python version is far too slow
%cd monotonic_align
!mkdir -p monotonic_align
!python setup.py build_ext --inplace
%cd /content/finetune-hf-vits
print("\nrecipe ready")

In [ ]:
# 3 · Log in to Hugging Face (paste a token with WRITE permission)
#    create one: https://huggingface.co/settings/tokens  (type: write)
from huggingface_hub import notebook_login, whoami
notebook_login()
WHO = whoami()
HF_USER = WHO["name"]
print("logged in as:", HF_USER)

In [ ]:
# 4 · Build a *training-ready* Khmer checkpoint
#     facebook/mms-tts-khm ships the generator only; fine-tuning is a GAN loop and needs
#     the converted discriminator too. There is no ready-made Khmer train checkpoint,
#     so this step creates one (do it once; it is pushed to YOUR hub).
import subprocess
LANG = "khm"
TRAIN_CKPT = f"{HF_USER}/mms-tts-khm-train"

cmd = ["python", "convert_original_discriminator_checkpoint.py",
       "--language_code", LANG,
       "--pytorch_dump_folder_path", "./mms-tts-khm-train",
       "--push_to_hub", TRAIN_CKPT]
print("$", " ".join(cmd))
r = subprocess.run(cmd)
print("\nconverter exit code:", r.returncode)
print("training checkpoint:", TRAIN_CKPT)


## 5 · The dataset — pick ONE of the two options below

**Option A** — your own voice (best; do the prep locally with
`khmer-colab/prepare_khmer_dataset.py`, then upload the resulting
`khmer-dataset.zip` here).

**Option B** — the free OpenSLR Khmer corpus (a general male Khmer voice, not yours).
Good for testing the whole pipeline end-to-end before you record anything.


In [ ]:
# 5A · Upload YOUR prepared dataset (khmer-dataset.zip from prepare_khmer_dataset.py)
import os, subprocess
from google.colab import files

DATA_REPO = f"{HF_USER}/khmer-voice-data"     # where the dataset goes on the Hub
USE_OWN_DATA = True                           # set False to use Option B instead

if USE_OWN_DATA:
    up = files.upload()                       # choose khmer-dataset.zip
    zp = list(up)[0]
    subprocess.run(["rm", "-rf", "/content/khmer-dataset"], check=False)
    subprocess.run(["unzip", "-q", "-o", os.path.join("/content", zp),
                    "-d", "/content/khmer-dataset"], check=True)
    # the zip may contain the folder itself — flatten if so
    inner = "/content/khmer-dataset/khmer-dataset"
    if os.path.isdir(inner):
        subprocess.run(["bash", "-c",
                        "mv /content/khmer-dataset/khmer-dataset/* /content/khmer-dataset/"],
                       check=False)
    audio_dir = "/content/khmer-dataset/audio"
    print("clips:", len(os.listdir(audio_dir)) if os.path.isdir(audio_dir) else "??")
    rep = "/content/khmer-dataset/report.txt"
    if os.path.exists(rep):
        print(open(rep, encoding="utf-8").read()[:600])
else:
    print("Using OpenSLR Khmer instead (continue to the next cell).")


In [ ]:
# 5B · (alternative) Download the free OpenSLR Khmer set  [CC BY-SA 4.0]
#      ~866 MB, transcribed: line_index.tsv + wavs/. Multi-speaker male Khmer —
#      good for a first test, but a fine-tune on many speakers sounds less personal.
if not USE_OWN_DATA:
    !cd /content && wget -q --show-progress https://www.openslr.org/resources/42/km_kh_male.zip
    !cd /content && rm -rf km_kh_male && unzip -q -o km_kh_male.zip -d km_kh_male
    !ls /content/km_kh_male | head
    !head -3 /content/km_kh_male/line_index.tsv
else:
    print("skipped (using your own recordings)")

In [ ]:
# 6 · Turn that folder into a Hugging Face dataset and push it to your hub
import os, json
from datasets import Dataset, Audio, Features, Value

ROOT = "/content/khmer-dataset" if USE_OWN_DATA else "/content/km_kh_male"

def load_own(root):
    meta = os.path.join(root, "metadata.jsonl")
    rows = [json.loads(l) for l in open(meta, encoding="utf-8") if l.strip()]
    for r in rows:
        r["audio"] = os.path.join(root, r["audio"])
    return rows

def load_openslr(root):
    rows = []
    idx = os.path.join(root, "line_index.tsv")
    for line in open(idx, encoding="utf-8"):
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 2:
            continue
        wav = os.path.join(root, "wavs", parts[0] + ".wav")
        if os.path.exists(wav):
            rows.append({"audio": wav, "text": parts[1]})
    return rows

rows = load_own(ROOT) if USE_OWN_DATA else load_openslr(ROOT)
print("rows:", len(rows))
print("first:", rows[0]["text"][:70] if rows else "(none)")

ds = Dataset.from_list(rows).cast_column("audio", Audio(sampling_rate=16000))
print(ds)
print("total minutes:", round(sum(len(a["array"]) / a["sampling_rate"] for a in ds["audio"]) / 60, 1))

ds.push_to_hub(DATA_REPO, private=False)
DATASET_REPO = DATA_REPO
print("\ndataset on the hub:", DATASET_REPO)

## 7 · The training config (tuned for the free T4, 16 GB)

The fields below are the ones the recipe documents. Three things matter for a free
Colab session:

* **checkpoint often** — every save is pushed to your Hugging Face repo, so a Colab
  disconnect costs you minutes, not hours. The cell below works out a sensible
  interval from your dataset size.
* **batch 8 + accumulate 4** — an effective batch of 32 that fits a T4 comfortably
  with fp16, the same shape used by the Khmer TTS project this kit borrows from.
* **epochs depend on how much audio you have** — a few hundred clips want many
  epochs; several hours of audio wants few (20 is plenty).


In [ ]:
# 7 · Write the config. Change VOICE_NAME if you like.
VOICE_NAME = "khmer-voice"      # your model name on the hub

BATCH = 8                       # T4-safe; paired with accumulation below
ACCUM = 4                       # effective batch = 8 x 4 = 32
EPOCHS = 200                    # ~150-400 clips -> many epochs.
                                # Several hours of audio -> 20 is plenty.

# checkpoint every ~half epoch (pushed to your hub): a Colab drop costs minutes
try:
    steps_per_epoch = max(1, -(-len(ds) // (BATCH * ACCUM)))     # ceil
except Exception:
    steps_per_epoch = 100
SAVE_EVERY = max(25, min(200, steps_per_epoch // 2))
print(f"clips: {globals().get('len', lambda x: '?')(ds) if 'ds' in globals() else '?'}"
      f" | steps/epoch ~{steps_per_epoch} | checkpoint every {SAVE_EVERY} steps")

config = {
    "project_name": VOICE_NAME,
    "push_to_hub": True,
    "hub_model_id": f"{HF_USER}/{VOICE_NAME}",
    "overwrite_output_dir": False,          # keep checkpoints -> resume works
    "output_dir": "./tmp/khmer_tts_finetuned",

    "dataset_name": DATASET_REPO,
    "dataset_config_name": None,
    "audio_column_name": "audio",
    "text_column_name": "text",
    "train_split_name": "train",
    "eval_split_name": "train",

    "max_duration_in_seconds": 20,
    "min_duration_in_seconds": 1.0,
    "max_tokens_length": 500,
    "full_generation_sample_text": "សួស្តីអ្នកទាំងអស់គ្នា",

    "model_name_or_path": TRAIN_CKPT,       # step 4's checkpoint

    "preprocessing_num_workers": 2,
    "do_train": True,
    "num_train_epochs": EPOCHS,
    "gradient_accumulation_steps": ACCUM,
    "gradient_checkpointing": False,
    "per_device_train_batch_size": BATCH,
    "learning_rate": 2e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_ratio": 0.01,
    "group_by_length": False,

    "do_eval": True,
    "eval_steps": SAVE_EVERY,
    "per_device_eval_batch_size": 8,
    "max_eval_samples": 25,
    "do_step_schedule_per_epoch": True,

    "save_steps": SAVE_EVERY,        # every checkpoint is pushed to your hub
    "save_total_limit": 3,
    "logging_steps": 25,

    "weight_disc": 3, "weight_fmaps": 1, "weight_gen": 1,
    "weight_kl": 1.5, "weight_duration": 1, "weight_mel": 35,

    "fp16": True,
    "seed": 456,
}
json.dump(config, open("/content/finetune-hf-vits/khmer_config.json", "w"), indent=2)
print("\nmodel will be pushed to:", config["hub_model_id"])
print("checkpoint every", SAVE_EVERY, "steps;", EPOCHS, "epochs max")


In [ ]:
# 8 · TRAIN  (watch the loss; the first audio samples appear after ~50 steps)
%cd /content/finetune-hf-vits
!accelerate launch run_vits_finetuning.py ./khmer_config.json

In [ ]:
# 9 · Colab dropped / crashed? Run this instead of cell 8: it resumes automatically.
#     (The recipe finds the newest checkpoint in output_dir and pushes to your hub.)
%cd /content/finetune-hf-vits
!accelerate launch run_vits_finetuning.py ./khmer_config.json --resume_from_checkpoint latest

In [ ]:
# 10 · Listen to your voice
import torch, scipy.io.wavfile as wav
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display

MODEL = f"{HF_USER}/{VOICE_NAME}"      # <- from cell 7 (or your hub repo id)
tok = AutoTokenizer.from_pretrained(MODEL)
model = VitsModel.from_pretrained(MODEL).to("cuda" if torch.cuda.is_available() else "cpu")

LINES = [
    "សូមស្វាគមន៍មកកាន់កម្មវិធីសៀវភៅសម្លេងខ្មែរ។",
    "នៅក្នុងសតវត្សទី៩ ព្រះរាជាណាចក្រខ្មែរបានកើតឡើង។",
    "ទឹកដីនេះមានប្រវត្តិយូរអង្វែង និងអរិយធម៌ដ៏អស្ចារ្យ។",
]
for i, line in enumerate(LINES, 1):
    with torch.no_grad():
        out = model(**tok(line, return_tensors="pt").to(model.device)).waveform
    sr = model.config.sampling_rate
    fn = f"/content/test_{i}.wav"
    wav.write(fn, sr, out.squeeze().cpu().numpy())
    print(fn, "-", line)
    display(Audio(fn))

## 11 · Use it in Sonora Khmer Studio

**A. As the offline engine** (recommended — your voice, no internet, no subscription):

1. In the studio folder run `install_khmer_tts.bat` → option **2**
   (adds the offline engine; it also downloads the two ready-made offline voices).
2. Point the **fine-tuned slot** at your model, either as an environment variable
   before starting:
   ```bat
   set KHMER_MMSFT_MODEL=YOURNAME/khmer-voice
   start.bat
   ```
   or by editing the `MMSFT_MODEL = ...` line in `pipeline/khmer_tts.py`.
3. Now the package speaks Khmer with **your voice**:
   * leave `tts_engine` as `"auto"` — your voice is used whenever the cloud
     engines are unavailable (no internet, Microsoft outage);
   * or set `"tts_engine": "mmsft"` in `pipeline/config.json` to always use it.

Engine names, in the order the package tries them by default:
`edge → mmsft → mms → voxcpm → gtts`.

**B. As a carrier for your RVC voice** — generate with this model (engine `mmsft`),
then keep using your existing RVC stage as before.

**Licence note:** MMS is **CC-BY-NC-4.0**, so models fine-tuned from it —
including this one — are **non-commercial**. For commercial audiobooks use your RVC
voice (trained on your own recordings) or VoxCPM2 (Apache-2.0).


---

## Appendix · where this workflow comes from

Three public sources shaped this kit — worth reading yourself:

| source | what it gave us |
|---|---|
| [fairseq — examples/mms](https://github.com/facebookresearch/fairseq/blob/main/examples/mms/README.md) | the original Meta MMS project: what the models are, the language list, and the official checkpoints (the *full* tarball also contains the discriminator, at `dl.fbaipublicfiles.com/mms/tts/full_model/<lang>.tar.gz`) |
| [PhonSobon/khmer-tts-kh](https://github.com/PhonSobon/khmer-tts-kh) | the reason step 4 exists: published Khmer checkpoints ship the generator only, and the GAN fine-tuning loop needs a converted discriminator. Also the source of the T4-safe batch/accumulation settings and the "filter to one speaker" advice |
| [KrorngAI/mms-tts-khm-finetuned](https://huggingface.co/KrorngAI/mms-tts-khm-finetuned) | proof that a community fine-tune of `facebook/mms-tts-khm` works as a drop-in `VitsModel` — and a ready-made Khmer voice you can use **without training anything** (it is the package's `mmsft` engine) |

Useful extras: the free [OpenSLR Khmer corpus](https://www.openslr.org/42/) (SLR42,
CC BY-SA 4.0) for a first test run, and `khmerttsopensource/khmer-tts` (a light
2-epoch fine-tune — mostly a curiosity, per its own model card).
